In [ ]:
df_train = pd.read_csv("segment_alerts_all_airports_train.csv", header=0)
df_test = pd.read_csv("dataset_set.csv", header=0)
print("Shape :", df.shape)
print("\nColonnes :")
print(df.columns.tolist())

print("\nTypes :")
print(df.dtypes)

print("\nAperçu :")
print(df.head())

print("\nValeurs manquantes :")
print(df.isna().sum())

print("\nNombre de doublons :", df.duplicated().sum())

In [ ]:
df_model = df.copy()

# ==========================================================
# 1. Transtypage
# ==========================================================
# Conversion de la date en datetime pandas.
# Les dates d'origine sont supposées en UTC.
# errors="coerce" transforme les dates invalides en NaT.
df_model["date"] = pd.to_datetime(df_model["date"], utc=True, errors="coerce")

# Conversion vers l'heure locale de Paris pour uniformiser l'analyse temporelle.
df_model["date"] = df_model["date"].dt.tz_convert("Europe/Paris")

# Liste des colonnes qui doivent être numériques.
num_cols = ["lon", "lat", "amplitude", "maxis", "dist", "azimuth"]

# Conversion explicite des colonnes numériques.
# Toute valeur non convertible devient NaN.
for col in num_cols:
    df_model[col] = pd.to_numeric(df_model[col], errors="coerce")

# Suppression des lignes où la date ou une variable numérique utile manque.
df_model = df_model.dropna(subset=["date"] + num_cols)

# Suppression des lignes où le type d'éclair (icloud) est manquant.
df_model = df_model.dropna(subset=["icloud"])


# ==========================================================
# 2. Nettoyage des valeurs aberrantes
# ==========================================================
# On retire les amplitudes extrêmes jugées aberrantes d'après l'EDA.
df_model = df_model[df_model["amplitude"].abs() < 300]

# On retire les éclairs trop éloignés.
df_model = df_model[df_model["dist"] < 200]

# On retire les erreurs de localisation trop grandes.
df_model = df_model[df_model["maxis"] < 10]


# ==========================================================
# 3. Variables de base
# ==========================================================
# On crée une variable plus parlante :
# 1 = cloud-ground (nuage-sol), 0 = intra-nuage.
df_model["is_cloud_ground"] = (~df_model["icloud"]).astype(int)

# On garde une copie texte du nom de l'aéroport.
# Elle servira pour les groupby même après le one-hot encoding final.
df_model["airport_name"] = df_model["airport"]


# ==========================================================
# 4. Tri chronologique AVANT la reconstruction des tempêtes
# ==========================================================
# On trie par aéroport puis par date pour garantir un ordre temporel correct.
df_model = df_model.sort_values(["airport_name", "date"]).reset_index(drop=True)

# ==========================================================
# 6. Features temporelles simples
# ==========================================================
# Extraction de composantes temporelles brutes.
df_model["hour"] = df_model["date"].dt.hour
df_model["month"] = df_model["date"].dt.month
df_model["dayofweek"] = df_model["date"].dt.dayofweek

# Encodage cyclique de l'heure.
# Permet au modèle de comprendre que 23h et 0h sont proches.
df_model["sin_hour"] = np.sin(2 * np.pi * df_model["hour"] / 24)
df_model["cos_hour"] = np.cos(2 * np.pi * df_model["hour"] / 24)

# Encodage cyclique du mois.
# Permet au modèle de comprendre que décembre et janvier sont proches.
df_model["sin_month"] = np.sin(2 * np.pi * df_model["month"] / 12)
df_model["cos_month"] = np.cos(2 * np.pi * df_model["month"] / 12)
